In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

files = [
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\xiaomixiaomi_products.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\zanoone_products - 2026-08-23.csv",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\arkaapi_first_50_pages.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\Digikala - Health & Beauty.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\Digikala - Home & Electronics - 2026-08-03.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\Khanoumi Data.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\SnappPay - All Cats - 2026-08-23.csv",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\SnappPay - Digital Accessories - 2026-08-23.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\SnappPay - Health & Beauty - 2026-08-23.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\SnappPay - Mobile.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\sormehshop_products.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\technolife_snapp_products_first_40_pages.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\Torob - Home & Electronics - 2026-08-22.xlsx",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\Torob Products - 17 June - Merged Files.csv",
    r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\TorobPay - Health & Beauty - 2026-08-23.xlsx",
]

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

OUTPUT_FOLDER = Path(r"C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\outputs")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [3]:
def read_data_file_robust(file_path):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        for encoding in ["utf-8-sig", "utf-8", "cp1256", "windows-1256", "latin1"]:
            try:
                return {"CSV": pd.read_csv(file_path, encoding=encoding, low_memory=False)}
            except UnicodeDecodeError:
                continue

        raise ValueError(f"Encoding مناسب برای {file_path.name} پیدا نشد.")

    if suffix in [".xlsx", ".xls"]:
        try:
            return pd.read_excel(file_path, sheet_name=None, engine="openpyxl")
        except Exception as excel_error:
            print(f"Excel reader ناموفق بود: {file_path.name}")
            print(f"جزئیات: {excel_error}")

            for encoding in ["utf-8-sig", "utf-8", "cp1256", "windows-1256", "latin1"]:
                try:
                    df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
                    print(f"فایل با موفقیت به‌عنوان CSV خوانده شد: {file_path.name}")
                    return {"CSV_detected": df}
                except Exception:
                    pass

            try:
                tables = pd.read_html(file_path)
                print(f"فایل با موفقیت به‌عنوان HTML Table خوانده شد: {file_path.name}")
                return {
                    f"HTML_Table_{i + 1}": table
                    for i, table in enumerate(tables)
                }
            except Exception:
                pass

            raise ValueError(
                f"فرمت واقعی فایل {file_path.name} قابل تشخیص نیست. "
                "ممکن است فایل خراب، دانلود ناقص یا فایل غیر Excel با پسوند xlsx باشد."
            )

    raise ValueError(f"فرمت فایل پشتیبانی نمی‌شود: {file_path.name}")

In [4]:
all_data = {}
read_errors = []

for file_path in files:
    file_name = Path(file_path).name

    try:
        all_data[file_name] = read_data_file_robust(file_path)
        print(f"✓ خوانده شد: {file_name}")

    except Exception as e:
        read_errors.append({
            "file_name": file_name,
            "error": str(e)
        })
        print(f"✗ خطا در خواندن: {file_name}")
        print(f"  {e}")

errors_df = pd.DataFrame(read_errors)

if not errors_df.empty:
    print("\nفایل‌های دارای خطا:")
    display(errors_df)

✓ خوانده شد: xiaomixiaomi_products.xlsx
✓ خوانده شد: zanoone_products - 2026-08-23.csv
✓ خوانده شد: arkaapi_first_50_pages.xlsx
✓ خوانده شد: Digikala - Health & Beauty.xlsx
✓ خوانده شد: Digikala - Home & Electronics - 2026-08-03.xlsx
✓ خوانده شد: Khanoumi Data.xlsx
✓ خوانده شد: SnappPay - All Cats - 2026-08-23.csv
Excel reader ناموفق بود: SnappPay - Digital Accessories - 2026-08-23.xlsx
جزئیات: File is not a zip file
فایل با موفقیت به‌عنوان CSV خوانده شد: SnappPay - Digital Accessories - 2026-08-23.xlsx
✓ خوانده شد: SnappPay - Digital Accessories - 2026-08-23.xlsx
✓ خوانده شد: SnappPay - Health & Beauty - 2026-08-23.xlsx
✓ خوانده شد: SnappPay - Mobile.xlsx
✓ خوانده شد: sormehshop_products.xlsx
✓ خوانده شد: technolife_snapp_products_first_40_pages.xlsx
✓ خوانده شد: Torob - Home & Electronics - 2026-08-22.xlsx
✓ خوانده شد: Torob Products - 17 June - Merged Files.csv
✓ خوانده شد: TorobPay - Health & Beauty - 2026-08-23.xlsx


سلول 4: توابع تمیزکاری
این توابع عنوان، برند، قیمت و وضعیت موجودی را برای استفادهٔ یکسان در همهٔ دیتاست‌ها آماده می‌کنند.

In [5]:
def clean_text(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = (
        value.replace("ي", "ی")
             .replace("ك", "ک")
             .replace("ۀ", "ه")
             .replace("\u200c", " ")
             .replace("\u200f", " ")
    )

    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA


def clean_price(series):
    cleaned = (
        series.astype("string")
              .str.replace(",", "", regex=False)
              .str.replace("٬", "", regex=False)
              .str.replace("ریال", "", regex=False)
              .str.replace("تومان", "", regex=False)
              .str.replace(r"[^\d.-]", "", regex=True)
              .replace("", pd.NA)
    )

    return pd.to_numeric(cleaned, errors="coerce")


def normalize_availability(series):
    text = series.astype("string").str.lower().str.strip()

    available_words = [
        "instock",
        "in stock",
        "available",
        "موجود",
        "true",
        "1"
    ]

    unavailable_words = [
        "outofstock",
        "out of stock",
        "unavailable",
        "ناموجود",
        "false",
        "0"
    ]

    result = pd.Series(pd.NA, index=series.index, dtype="boolean")

    result[text.isin(available_words)] = True
    result[text.isin(unavailable_words)] = False

    return result


def first_existing_column(df, column_names, default=pd.NA):
    for column in column_names:
        if column in df.columns:
            return df[column]

    return pd.Series(default, index=df.index)

سلول 5: استانداردسازی همهٔ منابع
این مهم‌ترین بخش است. ستون‌های هر فایل را به ستون‌های مشترک تبدیل می‌کند.

In [6]:
def standardize_dataset(df, file_name, sheet_name):
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == "object" or str(df[col].dtype).startswith("string"):
            df[col] = df[col].map(clean_text)

    file_lower = file_name.lower()

    master = pd.DataFrame(index=df.index)

    master["dataset"] = file_name
    master["sheet_name"] = sheet_name

    master["source"] = pd.NA
    master["source_id"] = pd.NA
    master["product_title"] = pd.NA
    master["product_subtitle"] = pd.NA
    master["brand"] = pd.NA
    master["category"] = pd.NA
    master["merchant_name"] = pd.NA
    master["price"] = pd.NA
    master["old_price"] = pd.NA
    master["availability_raw"] = pd.NA
    master["is_available"] = pd.Series(pd.NA, index=df.index, dtype="boolean")
    master["product_url"] = pd.NA
    master["image_url"] = pd.NA
    master["search_keyword"] = pd.NA
    master["extracted_at"] = pd.NA
    master["cash_back"] = pd.NA
    master["discount"] = pd.NA

    if "xiaomixiaomi" in file_lower:
        master["source"] = "xiaomixiaomi"
        master["source_id"] = first_existing_column(df, ["id"])
        master["product_title"] = first_existing_column(df, ["title"])
        master["category"] = first_existing_column(df, ["category"])
        master["price"] = first_existing_column(df, ["sale_price"])
        master["old_price"] = first_existing_column(df, ["regular_price"])
        master["availability_raw"] = first_existing_column(df, ["availability"])
        master["product_url"] = first_existing_column(df, ["link"])
        master["image_url"] = first_existing_column(df, ["image_link"])

    elif "zanoone" in file_lower:
        master["source"] = "zanoone"
        master["source_id"] = first_existing_column(df, ["id"])
        master["product_title"] = first_existing_column(df, ["title"])
        master["product_subtitle"] = first_existing_column(df, ["subtitle"])
        master["brand"] = first_existing_column(df, ["brand"])
        master["category"] = first_existing_column(df, ["category"])
        master["price"] = first_existing_column(df, ["sale_price"])
        master["old_price"] = first_existing_column(df, ["regular_price"])
        master["availability_raw"] = first_existing_column(df, ["availability"])
        master["product_url"] = first_existing_column(df, ["link"])
        master["image_url"] = first_existing_column(df, ["image_link"])

    elif "arkaapi" in file_lower:
        master["source"] = "arka"
        master["source_id"] = first_existing_column(df, ["id"])
        master["product_title"] = first_existing_column(df, ["title", "name"])
        master["product_subtitle"] = first_existing_column(df, ["subtitle"])
        master["brand"] = first_existing_column(df, ["brand"])
        master["category"] = first_existing_column(df, ["category"])
        master["price"] = first_existing_column(df, ["sale_price"])
        master["old_price"] = first_existing_column(df, ["regular_price"])
        master["availability_raw"] = first_existing_column(df, ["availability"])
        master["product_url"] = first_existing_column(df, ["link"])
        master["image_url"] = first_existing_column(df, ["picture"])

    elif "digikala" in file_lower:
        master["source"] = "digikala"
        master["source_id"] = pd.NA
        master["scrape_sequence"] = first_existing_column(df, ["Scrape Sequence"])
        master["product_title"] = first_existing_column(df, ["Product Title"])
        master["price"] = first_existing_column(df, ["Product Price"])
        master["category"] = first_existing_column(df, ["URL"])

    elif "khanoumi" in file_lower:
        master["source"] = "khanoumi"
        master["source_id"] = first_existing_column(df, ["page_unique"])
        master["product_title"] = first_existing_column(df, ["title", "h1"])
        master["product_subtitle"] = first_existing_column(df, ["subtitle"])
        master["brand"] = pd.NA
        master["category"] = first_existing_column(df, ["category_name"])
        master["price"] = first_existing_column(df, ["current_price"])
        master["old_price"] = first_existing_column(df, ["old_price"])
        master["availability_raw"] = first_existing_column(df, ["availability"])
        master["product_url"] = first_existing_column(df, ["page_url"])
        master["image_url"] = first_existing_column(df, ["image_link"])

    elif "snapp" in file_lower:
        master["source"] = "snapp_pay"
        master["source_id"] = first_existing_column(df, ["product_id"])
        master["product_title"] = first_existing_column(df, ["product_name"])
        master["merchant_name"] = first_existing_column(df, ["merchant_name"])
        master["price"] = first_existing_column(df, ["price"])
        master["old_price"] = first_existing_column(df, ["old_price"])
        master["product_url"] = first_existing_column(df, ["referral_link"])
        master["image_url"] = first_existing_column(df, ["image_link"])
        master["search_keyword"] = first_existing_column(df, ["keyword"])
        master["cash_back"] = first_existing_column(df, ["cash_back"])
        master["discount"] = first_existing_column(df, ["discount"])
        master["search_page"] = first_existing_column(df, ["page"])
        master["is_ad"] = first_existing_column(df, ["ad"])
        master["has_free_shipping"] = first_existing_column(df, ["has_free_shipping"])

    if "available" in df.columns:
        master["is_available"] = df["available"].astype("boolean")
        master["availability_raw"] = df["available"].astype("string")

    elif "sormehshop" in file_lower:
        master["source"] = "sormehshop"
        master["source_id"] = first_existing_column(df, ["id"])
        master["product_title"] = first_existing_column(df, ["title"])
        master["brand"] = first_existing_column(df, ["brand"])
        master["category"] = first_existing_column(df, ["category"])
        master["price"] = first_existing_column(df, ["sale_price"])
        master["old_price"] = first_existing_column(df, ["regular_price"])
        master["availability_raw"] = first_existing_column(df, ["availability"])
        master["product_url"] = first_existing_column(df, ["link"])
        master["image_url"] = first_existing_column(df, ["image_link"])

    elif "technolife" in file_lower:
        master["source"] = "technolife"
        master["source_id"] = first_existing_column(df, ["id"])
        master["product_title"] = first_existing_column(df, ["title"])
        master["brand"] = first_existing_column(df, ["brand"])
        master["category"] = first_existing_column(df, ["category"])
        master["price"] = first_existing_column(df, ["sale_price"])
        master["old_price"] = first_existing_column(df, ["regular_price"])
        master["availability_raw"] = first_existing_column(df, ["availability"])
        master["product_url"] = first_existing_column(df, ["link"])
        master["image_url"] = first_existing_column(df, ["image_link"])

    elif "torobpay" in file_lower:
        master["source"] = "torob_pay"
        master["product_title"] = first_existing_column(df, ["Product Title"])
        master["price"] = first_existing_column(df, ["Product Price"])
        master["search_keyword"] = first_existing_column(df, ["Search Query"])
        master["extracted_at"] = first_existing_column(df, ["Extraction Date"])

    elif "torob" in file_lower and "merged" in file_lower:
        master["source"] = "torob"
        master["product_title"] = first_existing_column(df, ["Product Title"])
        master["price"] = first_existing_column(df, ["Product Price"])
        master["merchant_name"] = first_existing_column(df, ["Shop URL"])
        master["product_url"] = first_existing_column(df, ["Shop URL"])

    elif "torob" in file_lower:
        master["source"] = "torob"
        master["product_title"] = first_existing_column(df, ["Product Title"])
        master["price"] = first_existing_column(df, ["Product Price"])
        master["search_keyword"] = first_existing_column(df, ["Search Query"])
        master["extracted_at"] = first_existing_column(df, ["Extraction Date"])

    else:
        master["source"] = "unknown"

    master["product_title"] = master["product_title"].map(clean_text)
    master["product_subtitle"] = master["product_subtitle"].map(clean_text)
    master["brand"] = master["brand"].map(clean_text)
    master["category"] = master["category"].map(clean_text)
    master["merchant_name"] = master["merchant_name"].map(clean_text)

    master["price"] = clean_price(master["price"])
    master["old_price"] = clean_price(master["old_price"])

    master["availability_raw"] = master["availability_raw"].map(clean_text)

    availability_from_text = normalize_availability(master["availability_raw"])

    master["is_available"] = master["is_available"].fillna(availability_from_text)

    master["discount_pct_calculated"] = np.where(
        (master["old_price"].notna()) &
        (master["old_price"] > 0) &
        (master["price"].notna()) &
        (master["price"] < master["old_price"]),
        ((master["old_price"] - master["price"]) / master["old_price"] * 100).round(2),
        np.nan
    )

    return master

سلول 6: ساخت جدول نهایی

In [7]:
master_frames = []

for file_name, sheets in all_data.items():
    for sheet_name, df in sheets.items():
        standardized_df = standardize_dataset(
            df=df,
            file_name=file_name,
            sheet_name=sheet_name
        )

        master_frames.append(standardized_df)

master_products = pd.concat(
    master_frames,
    ignore_index=True
)

master_products = master_products[
    master_products["product_title"].notna()
].copy()

master_products = master_products[
    ~master_products["product_title"]
    .str.lower()
    .eq("no products found")
].copy()

master_products["product_title_normalized"] = (
    master_products["product_title"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

display(master_products["source"].value_counts(dropna=False))

C:\Users\kourosh\AppData\Local\Temp\ipykernel_12864\1258768753.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_products = pd.concat(
C:\Users\kourosh\AppData\Local\Temp\ipykernel_12864\1258768753.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_products = pd.concat(


source
torob         569413
unknown       255011
snapp_pay     200628
technolife    158000
sormehshop      4089
Name: count, dtype: int64

In [8]:
def source_from_dataset(dataset_name):
    name = str(dataset_name).lower()

    if "xiaomixiaomi" in name:
        return "xiaomixiaomi"

    if "zanoone" in name:
        return "zanoone"

    if "arkaapi" in name:
        return "arka"

    if "digikala" in name:
        return "digikala"

    if "khanoumi" in name:
        return "khanoumi"

    if "sormehshop" in name:
        return "sormehshop"

    # باید قبل از snapp باشد، چون نام فایل Technolife شامل snapp است
    if "technolife" in name:
        return "technolife"

    # باید قبل از torob باشد
    if "torobpay" in name:
        return "torob_pay"

    if "torob" in name:
        return "torob"

    if "snapp" in name:
        return "snapp_pay"

    return "unknown"

In [9]:
master_products["source"] = (
    master_products["dataset"]
    .map(source_from_dataset)
)

source_counts = (
    master_products["source"]
    .value_counts(dropna=False)
    .rename("rows")
    .to_frame()
)

display(source_counts)

,rows
source,
torob,569413
snapp_pay,200628
technolife,158000
arka,130700
digikala,55339
khanoumi,49481
zanoone,17007
sormehshop,4089
xiaomixiaomi,2484


In [10]:
dataset_source_check = (
    master_products
    .groupby(["dataset", "source"])
    .size()
    .reset_index(name="rows")
    .sort_values(["source", "dataset"])
)

display(dataset_source_check)

,dataset,source,rows
9,arkaapi_first_50_pages.xlsx,arka,130700
0,Digikala - Health & Beauty.xlsx,digikala,5277
1,Digikala - Home & Electronics - 2026-08-03.xlsx,digikala,50062
2,Khanoumi Data.xlsx,khanoumi,49481
3,SnappPay - All Cats - 2026-08-23.csv,snapp_pay,158986
4,SnappPay - Digital Accessories - 2026-08-23.xlsx,snapp_pay,6930
5,SnappPay - Health & Beauty - 2026-08-23.xlsx,snapp_pay,19952
6,SnappPay - Mobile.xlsx,snapp_pay,14760
10,sormehshop_products.xlsx,sormehshop,4089
11,technolife_snapp_products_first_40_pages.xlsx,technolife,158000


سلول 7: کنترل کیفیت خروجی
این گزارش مشخص می‌کند از هر منبع چه تعداد ردیف باقی مانده و چند ردیف عنوان، قیمت، برند یا وضعیت موجودی دارند.

In [12]:
quality_report = (
    master_products
    .groupby("source", dropna=False)
    .agg(
        rows=("source", "size"),
        unique_titles=("product_title_normalized", "nunique"),
        titles_with_price=("price", lambda x: x.notna().sum()),
        titles_with_old_price=("old_price", lambda x: x.notna().sum()),
        titles_with_brand=("brand", lambda x: x.notna().sum()),
        titles_with_category=("category", lambda x: x.notna().sum()),
        titles_available=("is_available", lambda x: (x == True).sum()),
        titles_unavailable=("is_available", lambda x: (x == False).sum()),
        min_price=("price", "min"),
        median_price=("price", "median"),
        max_price=("price", "max")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(quality_report)

,source,rows,unique_titles,titles_with_price,titles_with_old_price,titles_with_brand,titles_with_category,titles_available,titles_unavailable,min_price,median_price,max_price
6,torob,569413,239431,569413,0.0,0,0,0,0,1,1200000.0,328400000333600000
3,snapp_pay,200628,132643,200628,70272.0,0,0,200628,0,10000,34800000.0,348000038880000
5,technolife,158000,155775,158000,158000.0,158000,158000,158000,0,8470,465300.0,1130252400
0,arka,130700,2607,114550,112800.0,1450,130700,0,0,73161,1321389.0,47855699
1,digikala,55339,9829,55339,0.0,0,55339,0,0,19900,14490000.0,689827500
2,khanoumi,49481,36072,49481,49481.0,0,49481,12313,37168,0,0.0,725671322
8,zanoone,17007,17005,17007,17007.0,17007,17007,3692,13315,0,2252500.0,148191400
4,sormehshop,4089,100,4089,4089.0,4060,4089,2349,1740,180000,1075000.0,21800000
7,xiaomixiaomi,2484,2484,2484,2313.0,0,2471,707,1777,0,3550000.0,299550000


سلول 8: بررسی ردیف‌های تکراری
داده‌های SnappPay به‌احتمال زیاد برای یک محصول در keyword یا صفحه‌های مختلف تکرار دارند. این کد فعلاً فقط تکرارها را گزارش می‌کند و هیچ داده‌ای را حذف نمی‌کند.

In [13]:
duplicate_report = (
    master_products
    .groupby(["source", "source_id", "merchant_name"], dropna=False)
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
    .sort_values("row_count", ascending=False)
)

print(f"تعداد گروه‌های تکراری: {len(duplicate_report):,}")
display(duplicate_report.head(20))

تعداد گروه‌های تکراری: 31,333


,source,source_id,merchant_name,row_count
2614,digikala,NaN,NaN,55339
375311,torob,NaN,NaN,14438
374058,torob,NaN,https://torob.com/shop/161663/پست-ابزار/محصولات/,1200
375234,torob,NaN,https://torob.com/shop/7834/ریو-کالا/محصولات/,1200
374353,torob,NaN,https://torob.com/shop/233482/لیا-سنتر/محصولات/,1200
375120,torob,NaN,https://torob.com/shop/5274/دارونت/محصولات/,1200
374929,torob,NaN,https://torob.com/shop/394282/نفس-لند/محصولات/,1200
374883,torob,NaN,https://torob.com/shop/378246/همنا-دات-آی-آر/محصولات/,1176
375088,torob,NaN,https://torob.com/shop/48/خانومی/محصولات/,1176
373932,torob,NaN,https://torob.com/shop/131100/آریشا-تایم/محصولات/,1176


In [14]:
snapp_duplicates = master_products[
    (master_products["source"] == "snapp_pay") &
    (master_products["source_id"].duplicated(keep=False))
].sort_values(["source_id", "merchant_name", "price"])

display(
    snapp_duplicates[
        [
            "source_id",
            "product_title",
            "merchant_name",
            "price",
            "old_price",
            "cash_back",
            "search_keyword",
            "is_available"
        ]
    ].head(30)
)

,source_id,product_title,merchant_name,price,old_price,cash_back,search_keyword,is_available
408199,00006a57b52005455c816c04c70c07dc,تبلت 10.9 اینچ سامسونگ مدل Galaxy Tab S10 Lite Wi-Fi با ظرفیت 128 گیگابایت و رم 6 گیگابایت، رزول...,ریمووین,866880000,<NA>,8.0,سامسونگ,True
442058,00006a57b52005455c816c04c70c07dc,تبلت 10.9 اینچ سامسونگ مدل Galaxy Tab S10 Lite Wi-Fi با ظرفیت 128 گیگابایت و رم 6 گیگابایت، رزول...,ریمووین,868000000,<NA>,8.0,سامسونگ,True
357161,0000d7c7f5ea9362fb10e88cca5f6450,قهوه هسته خرما پپتینا بسته 6 عددی,پپتینا,2500000,<NA>,11.0,قهوه,True
433461,0000d7c7f5ea9362fb10e88cca5f6450,قهوه هسته خرما پپتینا بسته 6 عددی,پپتینا,2500000,<NA>,11.0,قهوه,True
342220,000249586b5491df0bded854cd4c48b4,توستر 2 اسلایس وستینگ هاوس مدل WKTTF04BB,اس جی کالا,380000000,<NA>,NaN,توستر,True
349323,000249586b5491df0bded854cd4c48b4,توستر 2 اسلایس وستینگ هاوس مدل WKTTF04BB,اس جی کالا,380000000,<NA>,NaN,تستر,True
418078,0004d077514d8c23ad78ac6222d97a3c,ساعت هوشمند 7بند WS12 ULTRA2,لیراس,37810000,54010000.0,11.0,ساعت هوشمند,True
419068,0004d077514d8c23ad78ac6222d97a3c,ساعت هوشمند 7بند WS12 ULTRA2,لیراس,37810000,54010000.0,11.0,Smart watch,True
267221,000a00126ef0b6cc32d7e8940698277a,گوشی موبایل ریلمی مدل C75 دو سیم کارت ظرفیت 128 گیگابایت و رم 8 گیگابایت,پیشگامو,397800000,<NA>,10.0,ریلمی,True
448171,000a00126ef0b6cc32d7e8940698277a,گوشی موبایل ریلمی مدل C75 دو سیم کارت ظرفیت 128 گیگابایت و رم 8 گیگابایت,پیشگامو,468000000,<NA>,10.0,ریلمی,True


In [16]:
dataset_source_check = (
    master_products
    .groupby(["dataset", "source"])
    .size()
    .reset_index(name="rows")
    .sort_values(["source", "dataset"])
)

display(dataset_source_check)

,dataset,source,rows
9,arkaapi_first_50_pages.xlsx,arka,130700
0,Digikala - Health & Beauty.xlsx,digikala,5277
1,Digikala - Home & Electronics - 2026-08-03.xlsx,digikala,50062
2,Khanoumi Data.xlsx,khanoumi,49481
3,SnappPay - All Cats - 2026-08-23.csv,snapp_pay,158986
4,SnappPay - Digital Accessories - 2026-08-23.xlsx,snapp_pay,6930
5,SnappPay - Health & Beauty - 2026-08-23.xlsx,snapp_pay,19952
6,SnappPay - Mobile.xlsx,snapp_pay,14760
10,sormehshop_products.xlsx,sormehshop,4089
11,technolife_snapp_products_first_40_pages.xlsx,technolife,158000


In [17]:
parquet_ready = master_products.copy()

text_columns = [
    "source",
    "dataset",
    "sheet_name",
    "source_id",
    "product_title",
    "product_subtitle",
    "brand",
    "category",
    "merchant_name",
    "availability_raw",
    "product_url",
    "image_url",
    "search_keyword",
    "extracted_at",
    "discount",
    "product_title_normalized"
]

for column in text_columns:
    if column in parquet_ready.columns:
        parquet_ready[column] = parquet_ready[column].astype("string")

numeric_columns = [
    "price",
    "old_price",
    "cash_back",
    "discount_pct_calculated",
    "search_page"
]

for column in numeric_columns:
    if column in parquet_ready.columns:
        parquet_ready[column] = pd.to_numeric(
            parquet_ready[column],
            errors="coerce"
        )

boolean_columns = [
    "is_available",
    "is_ad"
]

for column in boolean_columns:
    if column in parquet_ready.columns:
        parquet_ready[column] = parquet_ready[column].astype("boolean")

print("نوع داده‌های اصلی پس از اصلاح:")
display(
    parquet_ready[
        [
            column for column in [
                "source",
                "source_id",
                "product_title",
                "price",
                "old_price",
                "is_available"
            ]
            if column in parquet_ready.columns
        ]
    ].dtypes.to_frame("data_type")
)

نوع داده‌های اصلی پس از اصلاح:


,data_type
source,string[python]
source_id,string[python]
product_title,string[python]
price,Int64
old_price,Float64
is_available,boolean


In [18]:
print("نوع ستون source_id:")
print(parquet_ready["source_id"].dtype)

print("\nنمونه شناسه‌ها:")
display(
    parquet_ready[
        ["source", "source_id", "product_title"]
    ].sample(15, random_state=42)
)

نوع ستون source_id:
string

نمونه شناسه‌ها:


,source,source_id,product_title
174750,digikala,<NA>,گوشت کوب برقی 1300 وات دسینی مدل DK-1100
731869,torob,<NA>,کرم پودر فول کاور نیولنسین NEWLLENCEIN 32h حجم 50ml شماره R3
1098118,torob,<NA>,شامپو رنگ سوبارو Subaru دوقلو اصل- رنگ مو مشکی
1172052,torob,<NA>,لپ تاپ گیمینگ اچ پی مدل Victus 15 FA2013DX پردازنده i5 13420H حافظه ۱۶ گیگابایت SSD ۲ ترابایت گر...
480141,technolife,TLP-488026,محافظ صفحه نمایش شیشه ای کوکونات مدل UD مناسب برای گوشی موبایل اپل iPhone 13 pro به همراه محافظ ...
680393,torob,<NA>,گوشت کوب برقی یونیک مدل PS811 تک کاره
775167,torob,<NA>,محافظ آنالوگ دسته ، روکش آنالوگ چلسی
1053345,torob,<NA>,محافظ صفحه نمایش گلس سامسونگ مدل S24 FE
914649,torob,<NA>,قرص زیر زبانی واناتونین ملاتونین نورم لایف 10 میلی گرم 60 عدد
738331,torob,<NA>,تین کلاینت اچ پی T755


In [19]:
parquet_path = OUTPUT_FOLDER / "master_products_raw_normalized.parquet"
csv_path = OUTPUT_FOLDER / "master_products_raw_normalized.csv"
report_path = OUTPUT_FOLDER / "data_quality_report_by_source.xlsx"

parquet_ready.to_parquet(
    parquet_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

quality_report.to_excel(
    report_path,
    index=False
)

parquet_ready.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("فایل‌ها با موفقیت ذخیره شدند:")
print(f"Parquet: {parquet_path}")
print(f"CSV: {csv_path}")
print(f"Quality report: {report_path}")

فایل‌ها با موفقیت ذخیره شدند:
Parquet: C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\outputs\master_products_raw_normalized.parquet
CSV: C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\outputs\master_products_raw_normalized.csv
Quality report: C:\Users\kourosh\Desktop\Snapp pay\Pricing Data\outputs\data_quality_report_by_source.xlsx


گرفتن خروجی ها برای مطمعن شدن از جامع بودن و درستی یکسان سازی:


In [20]:
source_dataset_report = (
    master_products
    .groupby(["source", "dataset"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source", "dataset"])
)

display(source_dataset_report)

,source,dataset,rows
0,arka,arkaapi_first_50_pages.xlsx,130700
1,digikala,Digikala - Health & Beauty.xlsx,5277
2,digikala,Digikala - Home & Electronics - 2026-08-03.xlsx,50062
3,khanoumi,Khanoumi Data.xlsx,49481
4,snapp_pay,SnappPay - All Cats - 2026-08-23.csv,158986
5,snapp_pay,SnappPay - Digital Accessories - 2026-08-23.xlsx,6930
6,snapp_pay,SnappPay - Health & Beauty - 2026-08-23.xlsx,19952
7,snapp_pay,SnappPay - Mobile.xlsx,14760
8,sormehshop,sormehshop_products.xlsx,4089
9,technolife,technolife_snapp_products_first_40_pages.xlsx,158000


2. گزارش کامل کیفیت داده
این مهم‌ترین خروجی برای گزارش مدیریتی است:

In [21]:
quality_report = (
    master_products
    .groupby("source", dropna=False)
    .agg(
        rows=("source", "size"),
        unique_titles=("product_title_normalized", "nunique"),
        rows_with_price=("price", lambda x: x.notna().sum()),
        rows_without_price=("price", lambda x: x.isna().sum()),
        zero_price_rows=("price", lambda x: (x == 0).sum()),
        rows_with_old_price=("old_price", lambda x: x.notna().sum()),
        rows_with_brand=("brand", lambda x: x.notna().sum()),
        rows_with_category=("category", lambda x: x.notna().sum()),
        rows_available=("is_available", lambda x: (x == True).sum()),
        rows_unavailable=("is_available", lambda x: (x == False).sum()),
        rows_unknown_availability=("is_available", lambda x: x.isna().sum()),
        min_price=("price", "min"),
        median_price=("price", "median"),
        max_price=("price", "max")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(quality_report)

,source,rows,unique_titles,rows_with_price,rows_without_price,zero_price_rows,rows_with_old_price,rows_with_brand,rows_with_category,rows_available,rows_unavailable,rows_unknown_availability,min_price,median_price,max_price
6,torob,569413,239431,569413,0,0,0.0,0,0,0,0,569413,1,1200000.0,328400000333600000
3,snapp_pay,200628,132643,200628,0,0,70272.0,0,0,200628,0,0,10000,34800000.0,348000038880000
5,technolife,158000,155775,158000,0,0,158000.0,158000,158000,158000,0,0,8470,465300.0,1130252400
0,arka,130700,2607,114550,16150,0,112800.0,1450,130700,0,0,130700,73161,1321389.0,47855699
1,digikala,55339,9829,55339,0,0,0.0,0,55339,0,0,55339,19900,14490000.0,689827500
2,khanoumi,49481,36072,49481,0,37143,49481.0,0,49481,12313,37168,0,0,0.0,725671322
8,zanoone,17007,17005,17007,0,1436,17007.0,17007,17007,3692,13315,0,0,2252500.0,148191400
4,sormehshop,4089,100,4089,0,0,4089.0,4060,4089,2349,1740,0,180000,1075000.0,21800000
7,xiaomixiaomi,2484,2484,2484,0,5,2313.0,0,2471,707,1777,0,0,3550000.0,299550000


3. بررسی قیمت‌های قابل‌استفاده
این خروجی مشخص می‌کند چند ردیف واقعاً قیمت معتبر دارند؛ یعنی قیمت نه خالی است و نه صفر.

In [22]:
products_with_valid_price = master_products[
    (master_products["price"].notna()) &
    (master_products["price"] > 0)
].copy()

valid_price_report = (
    products_with_valid_price
    .groupby("source")
    .agg(
        valid_price_rows=("price", "size"),
        unique_titles=("product_title_normalized", "nunique"),
        min_price=("price", "min"),
        median_price=("price", "median"),
        mean_price=("price", "mean"),
        max_price=("price", "max")
    )
    .reset_index()
    .sort_values("valid_price_rows", ascending=False)
)

print(f"تعداد کل ردیف‌ها در master_products: {len(master_products):,}")
print(f"تعداد کل ردیف‌های دارای قیمت معتبر: {len(products_with_valid_price):,}")

display(valid_price_report)

تعداد کل ردیف‌ها در master_products: 1,187,141
تعداد کل ردیف‌های دارای قیمت معتبر: 1,132,407


,source,valid_price_rows,unique_titles,min_price,median_price,mean_price,max_price
6,torob,569413,239431,1,1200000.0,1938692564960.272217,328400000333600000
3,snapp_pay,200628,132643,10000,34800000.0,2353204127.223383,348000038880000
5,technolife,158000,155775,8470,465300.0,14363537.522462,1130252400
0,arka,114550,2285,73161,1321389.0,2303912.78612,47855699
1,digikala,55339,9829,19900,14490000.0,33532756.776595,689827500
8,zanoone,15571,15569,700,2475000.0,4706463.415966,148191400
2,khanoumi,12338,10194,29900,1266500.0,20614603.747204,725671322
4,sormehshop,4089,100,180000,1075000.0,1520574.638298,21800000
7,xiaomixiaomi,2479,2479,89000,3560000.0,7872824.12263,299550000


4. بررسی تکرارهای واقعی SnappPay
این گزارش فقط SnappPay را بررسی می‌کند و دیگر Torob، Digikala یا شناسه‌های خالی وارد محاسبه نمی‌شوند.

In [23]:
snapp_products = master_products[
    (master_products["source"] == "snapp_pay") &
    (master_products["source_id"].notna())
].copy()

snapp_duplicate_report = (
    snapp_products
    .groupby(
        ["dataset", "source_id", "merchant_name"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
    .sort_values("row_count", ascending=False)
)

snapp_duplicate_summary = (
    snapp_duplicate_report
    .groupby("dataset")
    .agg(
        duplicate_groups=("source_id", "size"),
        repeated_rows=("row_count", "sum"),
        max_repetitions=("row_count", "max")
    )
    .reset_index()
    .sort_values("repeated_rows", ascending=False)
)

print(f"تعداد ردیف‌های SnappPay: {len(snapp_products):,}")
print(f"تعداد گروه‌های دارای تکرار: {len(snapp_duplicate_report):,}")

display(snapp_duplicate_summary)
display(snapp_duplicate_report.head(20))

تعداد ردیف‌های SnappPay: 200,628
تعداد گروه‌های دارای تکرار: 19,196


,dataset,duplicate_groups,repeated_rows,max_repetitions
0,SnappPay - All Cats - 2026-08-23.csv,13970,31353,20
3,SnappPay - Mobile.xlsx,2705,6717,11
1,SnappPay - Digital Accessories - 2026-08-23.xlsx,1956,4226,13
2,SnappPay - Health & Beauty - 2026-08-23.xlsx,565,1439,11


,dataset,source_id,merchant_name,row_count
29457,SnappPay - All Cats - 2026-08-23.csv,35b57e4e20565775d55e2a2a7e3de162,طهران نوآ,20
40610,SnappPay - All Cats - 2026-08-23.csv,498e8d2f2fa01132e392e2be9cd77276,دنیا کامپیوتر,20
96256,SnappPay - All Cats - 2026-08-23.csv,adbbc7636c5633e2ca512843d1aee5e2,آرکا شاپ,19
48610,SnappPay - All Cats - 2026-08-23.csv,5812a689d7848486f426a529baf3a36b,فلورمار شاپ,18
25963,SnappPay - All Cats - 2026-08-23.csv,2f6f1c3146061925b97d98dce2cc3435,رونیشاپ,18
133828,SnappPay - All Cats - 2026-08-23.csv,f217fe2aecaa748467006d5a7d4882c4,گروه استاپ,18
59688,SnappPay - All Cats - 2026-08-23.csv,6c12fd2b638fc0ec040d2c96ec3a3b7a,شارژی کالا,18
160,SnappPay - All Cats - 2026-08-23.csv,004be5388b3d097c1ac68efe53f1b90f,دنیا کامپیوتر,17
43905,SnappPay - All Cats - 2026-08-23.csv,4f7ee8312788bca26eaca5f9d913a5d4,عدلا,16
23886,SnappPay - All Cats - 2026-08-23.csv,2b8dca1f081b64f7fae56d653610c12a,انتخاب سنتر,14
